# Reinforcement Learning Fundamentals

**Module 14: Reinforcement Learning**  
**Objective**: Master RL from basics to deep RL algorithms

Reinforcement Learning:
- MDP (Markov Decision Process)
- Q-Learning and Value Iteration
- Deep Q-Network (DQN)
- Policy Gradients
- Actor-Critic Methods
- Experience Replay and Target Networks

## What You'll Learn
1. RL fundamentals (agent, environment, reward)
2. Bellman equations and value functions
3. Q-learning algorithm
4. DQN for continuous state spaces
5. Policy gradient methods (REINFORCE)
6. Actor-Critic architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List, Optional
from collections import deque, namedtuple
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## 1. RL Framework

**Core Components**:
- **Agent**: Learner/decision maker
- **Environment**: What agent interacts with
- **State** ($s$): Current situation
- **Action** ($a$): What agent can do
- **Reward** ($r$): Feedback signal

**Goal**: Learn policy $\pi(a|s)$ that maximizes expected return:

$$G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1}$$

Where $\gamma \in [0,1]$ is discount factor

In [ ]:
def visualize_rl_framework():
    """Visualize RL agent-environment interaction."""
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # 1. Agent-Environment Loop
    ax1 = axes[0]
    ax1.axis('off')
    ax1.set_title('Agent-Environment Interaction', fontsize=14, weight='bold')
    ax1.set_xlim(0, 1)
    ax1.set_ylim(0, 1)
    
    # Agent box
    rect_agent = plt.Rectangle((0.15, 0.55), 0.25, 0.25, color='lightblue',
                               ec='black', linewidth=3)
    ax1.add_patch(rect_agent)
    ax1.text(0.275, 0.675, 'AGENT', ha='center', va='center',
            fontsize=14, weight='bold')
    ax1.text(0.275, 0.62, 'Policy π(a|s)', ha='center', va='center',
            fontsize=10, style='italic')
    
    # Environment box
    rect_env = plt.Rectangle((0.6, 0.55), 0.25, 0.25, color='lightgreen',
                            ec='black', linewidth=3)
    ax1.add_patch(rect_env)
    ax1.text(0.725, 0.675, 'ENVIRONMENT', ha='center', va='center',
            fontsize=14, weight='bold')
    
    # Action arrow (Agent -> Environment)
    ax1.annotate('', xy=(0.6, 0.7), xytext=(0.42, 0.7),
                arrowprops=dict(arrowstyle='->', lw=3, color='red'))
    ax1.text(0.51, 0.75, 'Action aₜ', ha='center', fontsize=11,
            weight='bold', color='red')
    
    # State arrow (Environment -> Agent, top)
    ax1.annotate('', xy=(0.275, 0.82), xytext=(0.725, 0.82),
                arrowprops=dict(arrowstyle='->', lw=3, color='blue',
                              connectionstyle='arc3,rad=.3'))
    ax1.text(0.5, 0.9, 'State sₜ₊₁', ha='center', fontsize=11,
            weight='bold', color='blue')
    
    # Reward arrow (Environment -> Agent, bottom)
    ax1.annotate('', xy=(0.4, 0.58), xytext=(0.6, 0.58),
                arrowprops=dict(arrowstyle='->', lw=3, color='green',
                              connectionstyle='arc3,rad=-.3'))
    ax1.text(0.5, 0.48, 'Reward rₜ₊₁', ha='center', fontsize=11,
            weight='bold', color='green')
    
    # Time annotation
    ax1.text(0.5, 0.15, 't → t+1 → t+2 → ...', ha='center',
            fontsize=12, style='italic',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # 2. MDP Diagram
    ax2 = axes[1]
    ax2.axis('off')
    ax2.set_title('Markov Decision Process (MDP)', fontsize=14, weight='bold')
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    
    # States
    states = [
        (0.2, 0.7, 's₁', 'lightblue'),
        (0.5, 0.7, 's₂', 'lightgreen'),
        (0.8, 0.7, 's₃', 'lightyellow'),
        (0.35, 0.45, 's₄', 'lightcoral')
    ]
    
    for x, y, label, color in states:
        circle = plt.Circle((x, y), 0.06, color=color, ec='black', linewidth=2)
        ax2.add_patch(circle)
        ax2.text(x, y, label, ha='center', va='center',
                fontsize=11, weight='bold')
    
    # Transitions with probabilities
    transitions = [
        ((0.26, 0.7), (0.44, 0.7), 'a₁, p=0.8', 'blue'),
        ((0.56, 0.7), (0.74, 0.7), 'a₂, p=0.9', 'red'),
        ((0.23, 0.64), (0.32, 0.51), 'a₃, p=0.7', 'green'),
    ]
    
    for start, end, label, color in transitions:
        ax2.annotate('', xy=end, xytext=start,
                    arrowprops=dict(arrowstyle='->', lw=2, color=color))
        mid_x = (start[0] + end[0]) / 2
        mid_y = (start[1] + end[1]) / 2 + 0.05
        ax2.text(mid_x, mid_y, label, ha='center', fontsize=8,
                color=color, weight='bold')
    
    # Rewards
    ax2.text(0.8, 0.62, 'r=+10', fontsize=10, weight='bold',
            color='green',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
    ax2.text(0.35, 0.37, 'r=-5', fontsize=10, weight='bold',
            color='red',
            bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))
    
    # Legend
    ax2.text(0.5, 0.15, 'MDP = (S, A, P, R, γ)', ha='center',
            fontsize=11, style='italic',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax2.text(0.5, 0.05, 'S: states, A: actions, P: transition prob, R: rewards',
            ha='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    print("Reinforcement Learning Framework:")
    print("="*70)
    
    print("\nKEY CONCEPTS:")
    print("  • Agent observes state sₜ from environment")
    print("  • Agent selects action aₜ based on policy π(a|s)")
    print("  • Environment transitions to new state sₜ₊₁")
    print("  • Agent receives reward rₜ₊₁")
    print("  • Goal: Maximize cumulative discounted reward")
    
    print("\nMARKOV PROPERTY:")
    print("  P(sₜ₊₁, rₜ₊₁ | sₜ, aₜ, ..., s₀, a₀) = P(sₜ₊₁, rₜ₊₁ | sₜ, aₜ)")
    print("  Future depends only on present, not history")
    
    print("\nRETURN (Discounted sum of rewards):")
    print("  Gₜ = rₜ₊₁ + γ·rₜ₊₂ + γ²·rₜ₊₃ + ...")
    print("  γ=0: Myopic (only immediate reward)")
    print("  γ=1: Far-sighted (all future rewards equal)")
    print("  γ=0.99: Typical (slight preference for immediate)")

visualize_rl_framework()

## 2. Value Functions

**State-value function** $V^\pi(s)$: Expected return from state $s$ following policy $\pi$

$$V^\pi(s) = \mathbb{E}_\pi[G_t | S_t = s]$$

**Action-value function** $Q^\pi(s,a)$: Expected return from state $s$, taking action $a$, then following $\pi$

$$Q^\pi(s,a) = \mathbb{E}_\pi[G_t | S_t = s, A_t = a]$$

**Bellman Equation**:

$$V^\pi(s) = \sum_a \pi(a|s) \sum_{s',r} p(s',r|s,a)[r + \gamma V^\pi(s')]$$

In [ ]:
class GridWorld:
    """
    Simple grid world environment for RL.
    
    Agent moves in 4x4 grid:
    - Start: top-left
    - Goal: bottom-right (+10 reward)
    - Pit: (1,1) (-5 reward)
    - Step: -1 reward
    """
    
    def __init__(self, size: int = 4):
        self.size = size
        self.n_states = size * size
        self.n_actions = 4  # up, right, down, left
        
        self.goal_state = (size-1, size-1)
        self.pit_state = (1, 1)
        
        self.reset()
    
    def reset(self) -> Tuple[int, int]:
        """Reset to start state."""
        self.state = (0, 0)
        return self.state
    
    def step(self, action: int) -> Tuple[Tuple[int, int], float, bool]:
        """
        Take action in environment.
        
        Args:
            action: 0=up, 1=right, 2=down, 3=left
            
        Returns:
            next_state: New state
            reward: Reward received
            done: Whether episode ended
        """
        row, col = self.state
        
        # Move
        if action == 0:  # up
            row = max(0, row - 1)
        elif action == 1:  # right
            col = min(self.size - 1, col + 1)
        elif action == 2:  # down
            row = min(self.size - 1, row + 1)
        elif action == 3:  # left
            col = max(0, col - 1)
        
        self.state = (row, col)
        
        # Reward
        if self.state == self.goal_state:
            reward = 10.0
            done = True
        elif self.state == self.pit_state:
            reward = -5.0
            done = True
        else:
            reward = -1.0
            done = False
        
        return self.state, reward, done
    
    def state_to_index(self, state: Tuple[int, int]) -> int:
        """Convert (row, col) to flat index."""
        return state[0] * self.size + state[1]
    
    def index_to_state(self, index: int) -> Tuple[int, int]:
        """Convert flat index to (row, col)."""
        return (index // self.size, index % self.size)

# Test environment
env = GridWorld(size=4)

print("GridWorld Environment:")
print("="*70)
print(f"Size: {env.size}x{env.size}")
print(f"States: {env.n_states}")
print(f"Actions: {env.n_actions} (up, right, down, left)")
print(f"Start: (0, 0)")
print(f"Goal: {env.goal_state} (+10 reward)")
print(f"Pit: {env.pit_state} (-5 reward)")
print(f"Step cost: -1")

# Sample episode
print("\nSample Episode:")
state = env.reset()
print(f"  Start: {state}")

actions = [1, 1, 2, 2]  # right, right, down, down
action_names = ['up', 'right', 'down', 'left']

for action in actions:
    state, reward, done = env.step(action)
    print(f"  Action: {action_names[action]:6s} → State: {state}, Reward: {reward:+.1f}, Done: {done}")
    if done:
        break

## 3. Q-Learning

**Q-Learning**: Off-policy TD control algorithm

**Update rule**:

$$Q(s,a) \leftarrow Q(s,a) + \alpha [r + \gamma \max_{a'} Q(s',a') - Q(s,a)]$$

Where:
- $\alpha$: Learning rate
- $r$: Reward received
- $s'$: Next state
- $\max_{a'} Q(s',a')$: Best Q-value in next state

**Key property**: Converges to optimal Q* with appropriate $\alpha$ and exploration

In [ ]:
class QLearningAgent:
    """
    Q-Learning agent for tabular environments.
    """
    
    def __init__(self, n_states: int, n_actions: int,
                 learning_rate: float = 0.1,
                 gamma: float = 0.99,
                 epsilon: float = 0.1):
        
        self.n_states = n_states
        self.n_actions = n_actions
        self.lr = learning_rate
        self.gamma = gamma
        self.epsilon = epsilon
        
        # Q-table: Q(s, a)
        self.q_table = np.zeros((n_states, n_actions))
    
    def get_action(self, state_idx: int, training: bool = True) -> int:
        """
        Epsilon-greedy action selection.
        
        Args:
            state_idx: Current state index
            training: If True, use epsilon-greedy; else greedy
            
        Returns:
            action: Selected action
        """
        if training and np.random.random() < self.epsilon:
            # Explore: random action
            return np.random.randint(self.n_actions)
        else:
            # Exploit: greedy action
            return np.argmax(self.q_table[state_idx])
    
    def update(self, state_idx: int, action: int,
               reward: float, next_state_idx: int, done: bool):
        """
        Q-learning update.
        
        Q(s,a) ← Q(s,a) + α[r + γ·max Q(s',a') - Q(s,a)]
        """
        current_q = self.q_table[state_idx, action]
        
        if done:
            # Terminal state: no future rewards
            target_q = reward
        else:
            # Best Q-value in next state
            max_next_q = np.max(self.q_table[next_state_idx])
            target_q = reward + self.gamma * max_next_q
        
        # TD error
        td_error = target_q - current_q
        
        # Update Q-value
        self.q_table[state_idx, action] += self.lr * td_error
        
        return td_error

def train_q_learning(env: GridWorld, agent: QLearningAgent,
                     n_episodes: int = 500) -> List[float]:
    """
    Train Q-learning agent.
    
    Returns:
        episode_rewards: List of total rewards per episode
    """
    episode_rewards = []
    
    for episode in range(n_episodes):
        state = env.reset()
        state_idx = env.state_to_index(state)
        
        total_reward = 0
        done = False
        steps = 0
        max_steps = 100
        
        while not done and steps < max_steps:
            # Select action
            action = agent.get_action(state_idx, training=True)
            
            # Take action
            next_state, reward, done = env.step(action)
            next_state_idx = env.state_to_index(next_state)
            
            # Update Q-table
            agent.update(state_idx, action, reward, next_state_idx, done)
            
            total_reward += reward
            state_idx = next_state_idx
            steps += 1
        
        episode_rewards.append(total_reward)
        
        # Print progress
        if (episode + 1) % 100 == 0:
            avg_reward = np.mean(episode_rewards[-100:])
            print(f"Episode {episode+1}/{n_episodes}, Avg Reward (last 100): {avg_reward:.2f}")
    
    return episode_rewards

# Train agent
print("\nTraining Q-Learning Agent:")
print("="*70)

env = GridWorld(size=4)
agent = QLearningAgent(
    n_states=env.n_states,
    n_actions=env.n_actions,
    learning_rate=0.1,
    gamma=0.99,
    epsilon=0.1
)

rewards = train_q_learning(env, agent, n_episodes=500)

# Plot learning curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.set_title('Q-Learning: Episode Rewards', fontsize=12, weight='bold')
ax1.plot(rewards, alpha=0.3, label='Raw')

# Moving average
window = 50
if len(rewards) >= window:
    moving_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax1.plot(range(window-1, len(rewards)), moving_avg, 'r-',
            linewidth=2, label=f'{window}-episode MA')

ax1.set_xlabel('Episode')
ax1.set_ylabel('Total Reward')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Visualize learned policy
ax2 = axes[1]
ax2.set_title('Learned Policy (Q-Table)', fontsize=12, weight='bold')
ax2.set_xlim(-0.5, 3.5)
ax2.set_ylim(-0.5, 3.5)
ax2.set_aspect('equal')
ax2.invert_yaxis()

action_arrows = [
    (0, -0.3, '^'),  # up
    (0.3, 0, '>'),   # right
    (0, 0.3, 'v'),   # down
    (-0.3, 0, '<')   # left
]

for i in range(4):
    for j in range(4):
        state_idx = i * 4 + j
        
        # Color based on value
        max_q = np.max(agent.q_table[state_idx])
        color = plt.cm.RdYlGn((max_q + 20) / 30)  # Normalize for coloring
        
        rect = plt.Rectangle((j-0.4, i-0.4), 0.8, 0.8, 
                            facecolor=color, edgecolor='black', linewidth=2)
        ax2.add_patch(rect)
        
        # Best action arrow
        if (i, j) != env.goal_state and (i, j) != env.pit_state:
            best_action = np.argmax(agent.q_table[state_idx])
            dx, dy, marker = action_arrows[best_action]
            ax2.text(j, i, marker, ha='center', va='center',
                    fontsize=20, weight='bold')
        
        # Special states
        if (i, j) == env.goal_state:
            ax2.text(j, i, 'GOAL', ha='center', va='center',
                    fontsize=10, weight='bold', color='green')
        elif (i, j) == env.pit_state:
            ax2.text(j, i, 'PIT', ha='center', va='center',
                    fontsize=10, weight='bold', color='red')

ax2.set_xticks(range(4))
ax2.set_yticks(range(4))
ax2.set_xlabel('Column')
ax2.set_ylabel('Row')

plt.tight_layout()
plt.show()

print("\nFinal Q-Table (sample):")
print("State (0,0):")
print(f"  Q-values: {agent.q_table[0]}")
print(f"  Best action: {np.argmax(agent.q_table[0])} ({'up right down left'.split()[np.argmax(agent.q_table[0])]})")

## 4. Deep Q-Network (DQN)

**Problem**: Q-table doesn't scale to large/continuous state spaces

**Solution**: Approximate Q(s,a) with neural network

**DQN innovations**:
1. **Experience Replay**: Store transitions, sample random minibatches
2. **Target Network**: Separate network for computing targets (stabilizes learning)

**Loss**:

$$L(\theta) = \mathbb{E}_{(s,a,r,s')\sim D}[(r + \gamma \max_{a'} Q_{\text{target}}(s',a') - Q(s,a;\theta))^2]$$

In [ ]:
# Experience replay buffer
Transition = namedtuple('Transition', ('state', 'action', 'reward', 'next_state', 'done'))

class ReplayBuffer:
    """
    Experience replay buffer for DQN.
    """
    
    def __init__(self, capacity: int = 10000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, *args):
        """Save a transition."""
        self.buffer.append(Transition(*args))
    
    def sample(self, batch_size: int) -> List[Transition]:
        """Sample random batch."""
        return random.sample(self.buffer, batch_size)
    
    def __len__(self):
        return len(self.buffer)

class DQN(nn.Module):
    """
    Deep Q-Network.
    """
    
    def __init__(self, state_dim: int, n_actions: int, hidden_dim: int = 128):
        super().__init__()
        
        self.network = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, state_dim) states
            
        Returns:
            q_values: (batch, n_actions) Q-values for each action
        """
        return self.network(x)

class DQNAgent:
    """
    DQN Agent with experience replay and target network.
    """
    
    def __init__(self, state_dim: int, n_actions: int,
                 learning_rate: float = 1e-3,
                 gamma: float = 0.99,
                 epsilon_start: float = 1.0,
                 epsilon_end: float = 0.01,
                 epsilon_decay: float = 0.995,
                 buffer_size: int = 10000,
                 batch_size: int = 64,
                 target_update: int = 10):
        
        self.n_actions = n_actions
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update = target_update
        
        # Epsilon decay
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        
        # Q-network and target network
        self.q_network = DQN(state_dim, n_actions).to(device)
        self.target_network = DQN(state_dim, n_actions).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()
        
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=learning_rate)
        self.replay_buffer = ReplayBuffer(buffer_size)
        
        self.steps_done = 0
    
    def get_action(self, state: np.ndarray, training: bool = True) -> int:
        """
        Epsilon-greedy action selection.
        """
        if training and np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        else:
            with torch.no_grad():
                state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
                q_values = self.q_network(state_t)
                return q_values.argmax().item()
    
    def update(self):
        """
        Update Q-network using experience replay.
        """
        if len(self.replay_buffer) < self.batch_size:
            return 0.0
        
        # Sample batch
        transitions = self.replay_buffer.sample(self.batch_size)
        batch = Transition(*zip(*transitions))
        
        # Convert to tensors
        state_batch = torch.FloatTensor(np.array(batch.state)).to(device)
        action_batch = torch.LongTensor(batch.action).unsqueeze(1).to(device)
        reward_batch = torch.FloatTensor(batch.reward).to(device)
        next_state_batch = torch.FloatTensor(np.array(batch.next_state)).to(device)
        done_batch = torch.FloatTensor(batch.done).to(device)
        
        # Current Q-values
        current_q = self.q_network(state_batch).gather(1, action_batch)
        
        # Target Q-values (from target network)
        with torch.no_grad():
            next_q = self.target_network(next_state_batch).max(1)[0]
            target_q = reward_batch + (1 - done_batch) * self.gamma * next_q
        
        # Loss
        loss = F.smooth_l1_loss(current_q.squeeze(), target_q)
        
        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), 1.0)
        self.optimizer.step()
        
        return loss.item()
    
    def update_target_network(self):
        """Copy Q-network weights to target network."""
        self.target_network.load_state_dict(self.q_network.state_dict())
    
    def decay_epsilon(self):
        """Decay exploration rate."""
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

# Continuous state environment wrapper
class ContinuousGridWorld:
    """
    Grid world with continuous state representation.
    """
    
    def __init__(self, size: int = 4):
        self.env = GridWorld(size)
        self.state_dim = 2  # (row, col) normalized
        self.n_actions = 4
    
    def reset(self) -> np.ndarray:
        state = self.env.reset()
        return np.array(state, dtype=np.float32) / self.env.size
    
    def step(self, action: int) -> Tuple[np.ndarray, float, bool]:
        state, reward, done = self.env.step(action)
        state = np.array(state, dtype=np.float32) / self.env.size
        return state, reward, done

print("\nDQN Architecture:")
print("="*70)

env_continuous = ContinuousGridWorld(size=4)
dqn_agent = DQNAgent(
    state_dim=env_continuous.state_dim,
    n_actions=env_continuous.n_actions,
    learning_rate=1e-3,
    gamma=0.99,
    buffer_size=10000,
    batch_size=64
)

total_params = sum(p.numel() for p in dqn_agent.q_network.parameters())
print(f"Q-Network parameters: {total_params:,}")
print(f"\nQ-Network:")
print(dqn_agent.q_network)

print("\nKey DQN Components:")
print("  1. Q-Network: Approximates Q(s,a) with neural network")
print("  2. Target Network: Stabilizes training (updated every N steps)")
print("  3. Experience Replay: Breaks correlation, improves sample efficiency")
print("  4. Epsilon-greedy: Balances exploration vs exploitation")

## 5. Policy Gradients

**Policy Gradient** methods directly optimize policy $\pi_\theta(a|s)$

**Objective**: Maximize expected return

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[G(\tau)]$$

**REINFORCE** algorithm (Monte Carlo PG):

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau}[\sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t|s_t) G_t]$$

**Intuition**: Increase probability of actions that led to high returns

In [ ]:
class PolicyNetwork(nn.Module):
    """
    Policy network for REINFORCE.
    """
    
    def __init__(self, state_dim: int, n_actions: int, hidden_dim: int = 128):
        super().__init__()
        
        self.network = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions),
            nn.Softmax(dim=-1)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch, state_dim) states
            
        Returns:
            probs: (batch, n_actions) action probabilities
        """
        return self.network(x)

class REINFORCEAgent:
    """
    REINFORCE (Monte Carlo Policy Gradient) agent.
    """
    
    def __init__(self, state_dim: int, n_actions: int,
                 learning_rate: float = 1e-3,
                 gamma: float = 0.99):
        
        self.gamma = gamma
        self.n_actions = n_actions
        
        self.policy = PolicyNetwork(state_dim, n_actions).to(device)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=learning_rate)
        
        # Episode buffer
        self.log_probs = []
        self.rewards = []
    
    def get_action(self, state: np.ndarray) -> int:
        """
        Sample action from policy.
        """
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        probs = self.policy(state_t)
        
        # Sample action
        action_dist = torch.distributions.Categorical(probs)
        action = action_dist.sample()
        
        # Save log probability
        self.log_probs.append(action_dist.log_prob(action))
        
        return action.item()
    
    def store_reward(self, reward: float):
        """Store reward for current step."""
        self.rewards.append(reward)
    
    def update(self) -> float:
        """
        Update policy using REINFORCE.
        
        Computes returns and updates policy to increase
        probability of high-return actions.
        """
        # Compute returns (discounted cumulative rewards)
        returns = []
        G = 0
        for reward in reversed(self.rewards):
            G = reward + self.gamma * G
            returns.insert(0, G)
        
        returns = torch.FloatTensor(returns).to(device)
        
        # Normalize returns (reduces variance)
        returns = (returns - returns.mean()) / (returns.std() + 1e-9)
        
        # Policy gradient loss
        policy_loss = []
        for log_prob, G in zip(self.log_probs, returns):
            policy_loss.append(-log_prob * G)
        
        policy_loss = torch.stack(policy_loss).sum()
        
        # Optimize
        self.optimizer.zero_grad()
        policy_loss.backward()
        self.optimizer.step()
        
        # Clear episode buffer
        self.log_probs = []
        self.rewards = []
        
        return policy_loss.item()

def train_reinforce(env: ContinuousGridWorld, agent: REINFORCEAgent,
                    n_episodes: int = 1000) -> List[float]:
    """
    Train REINFORCE agent.
    """
    episode_rewards = []
    
    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0
        done = False
        steps = 0
        max_steps = 100
        
        # Collect episode
        while not done and steps < max_steps:
            action = agent.get_action(state)
            next_state, reward, done = env.step(action)
            
            agent.store_reward(reward)
            total_reward += reward
            
            state = next_state
            steps += 1
        
        # Update policy
        loss = agent.update()
        episode_rewards.append(total_reward)
        
        if (episode + 1) % 100 == 0:
            avg_reward = np.mean(episode_rewards[-100:])
            print(f"Episode {episode+1}/{n_episodes}, Avg Reward: {avg_reward:.2f}, Loss: {loss:.4f}")
    
    return episode_rewards

print("\nREINFORCE (Policy Gradient):")
print("="*70)

env_continuous = ContinuousGridWorld(size=4)
reinforce_agent = REINFORCEAgent(
    state_dim=env_continuous.state_dim,
    n_actions=env_continuous.n_actions,
    learning_rate=1e-3,
    gamma=0.99
)

print(f"Policy Network:")
print(reinforce_agent.policy)

print("\nKey Differences from Q-Learning:")
print("  • Q-Learning: Learn value function, derive policy")
print("  • Policy Gradient: Directly optimize policy")
print("  • REINFORCE: Monte Carlo (wait for full episode)")
print("  • Advantage: Can learn stochastic policies")
print("  • Disadvantage: High variance")

## 6. Actor-Critic

**Actor-Critic** combines value-based and policy-based methods

**Components**:
- **Actor**: Policy network $\pi_\theta(a|s)$
- **Critic**: Value network $V_\phi(s)$

**Advantage**:

$$A(s,a) = Q(s,a) - V(s) = r + \gamma V(s') - V(s)$$

**Actor update**: Policy gradient with advantage

$$\nabla_\theta J(\theta) = \mathbb{E}[\nabla_\theta \log \pi_\theta(a|s) A(s,a)]$$

**Critic update**: TD error

$$L(\phi) = (r + \gamma V_\phi(s') - V_\phi(s))^2$$

In [ ]:
class ActorCritic(nn.Module):
    """
    Actor-Critic network (shared backbone).
    """
    
    def __init__(self, state_dim: int, n_actions: int, hidden_dim: int = 128):
        super().__init__()
        
        # Shared layers
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU()
        )
        
        # Actor head (policy)
        self.actor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions),
            nn.Softmax(dim=-1)
        )
        
        # Critic head (value)
        self.critic = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            x: (batch, state_dim) states
            
        Returns:
            probs: (batch, n_actions) action probabilities
            value: (batch, 1) state value
        """
        features = self.shared(x)
        probs = self.actor(features)
        value = self.critic(features)
        return probs, value

class ActorCriticAgent:
    """
    Actor-Critic agent.
    """
    
    def __init__(self, state_dim: int, n_actions: int,
                 learning_rate: float = 1e-3,
                 gamma: float = 0.99):
        
        self.gamma = gamma
        self.n_actions = n_actions
        
        self.model = ActorCritic(state_dim, n_actions).to(device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=learning_rate)
        
        # Temporary storage
        self.log_prob = None
        self.value = None
    
    def get_action(self, state: np.ndarray) -> int:
        """
        Sample action from policy and compute value.
        """
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        probs, value = self.model(state_t)
        
        # Sample action
        action_dist = torch.distributions.Categorical(probs)
        action = action_dist.sample()
        
        # Save for update
        self.log_prob = action_dist.log_prob(action)
        self.value = value
        
        return action.item()
    
    def update(self, reward: float, next_state: np.ndarray, done: bool) -> Tuple[float, float]:
        """
        Update actor and critic.
        
        Returns:
            actor_loss: Policy loss
            critic_loss: Value loss
        """
        # Compute next value
        if done:
            next_value = torch.zeros(1, 1).to(device)
        else:
            next_state_t = torch.FloatTensor(next_state).unsqueeze(0).to(device)
            with torch.no_grad():
                _, next_value = self.model(next_state_t)
        
        # TD target
        target = reward + self.gamma * next_value
        
        # Advantage
        advantage = (target - self.value).detach()
        
        # Actor loss (policy gradient with advantage)
        actor_loss = -self.log_prob * advantage
        
        # Critic loss (TD error)
        critic_loss = F.smooth_l1_loss(self.value, target)
        
        # Total loss
        loss = actor_loss + critic_loss
        
        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        return actor_loss.item(), critic_loss.item()

print("\nActor-Critic Architecture:")
print("="*70)

ac_agent = ActorCriticAgent(
    state_dim=2,
    n_actions=4,
    learning_rate=1e-3
)

print("Actor-Critic Model:")
print(ac_agent.model)

# Test forward pass
test_state = torch.randn(1, 2).to(device)
with torch.no_grad():
    test_probs, test_value = ac_agent.model(test_state)

print(f"\nTest Forward Pass:")
print(f"  Input state: {test_state.shape}")
print(f"  Action probs: {test_probs.shape} (sums to {test_probs.sum().item():.4f})")
print(f"  State value: {test_value.shape} (value: {test_value.item():.4f})")

print("\nActor-Critic Benefits:")
print("  • Lower variance than REINFORCE (critic baseline)")
print("  • Online learning (TD updates, not Monte Carlo)")
print("  • Shared representation (actor and critic learn together)")
print("  • Foundation for advanced algorithms (A3C, PPO, SAC)")

## Summary

You've mastered RL fundamentals:
- ✅ RL framework (agent, environment, rewards)
- ✅ MDP and Bellman equations
- ✅ Q-Learning for tabular environments
- ✅ DQN with experience replay and target networks
- ✅ REINFORCE (policy gradients)
- ✅ Actor-Critic architecture

**Key Insights**:
1. **Q-Learning** learns action values, converges to optimal policy
2. **DQN** scales Q-learning to continuous spaces with neural networks
3. **Experience replay** breaks correlation, improves sample efficiency
4. **Policy gradients** directly optimize policy, handle stochastic policies
5. **Actor-Critic** combines best of both: low variance + online learning

**Algorithm Comparison**:

| Algorithm | Type | Variance | Sample Efficiency | On/Off Policy |
|-----------|------|----------|-------------------|---------------|
| Q-Learning | Value-based | Low | High | Off-policy |
| DQN | Value-based | Low | High | Off-policy |
| REINFORCE | Policy-based | High | Low | On-policy |
| Actor-Critic | Hybrid | Medium | Medium | On-policy |

**Applications**:
- Game playing (Atari, Go, Poker)
- Robotics control
- Autonomous driving
- Resource allocation
- RLHF for language models (next notebook!)

**Next Notebook**: Advanced RL (PPO, RLHF, reward modeling)

## Exercises

1. **Train DQN**: Implement full DQN training loop for CartPole
2. **Compare algorithms**: Train Q-Learning, DQN, REINFORCE, Actor-Critic on same environment
3. **Tune hyperparameters**: Experiment with learning rate, gamma, epsilon decay
4. **Implement Double DQN**: Reduce overestimation bias in DQN
5. **Add entropy regularization**: Encourage exploration in policy gradient methods